# Workspace Setup (Admin-only)

This notebook provisions shared resources for the course. Run it once per workspace before students begin.

**What it creates:**
1. Shared Unity Catalog namespace (`dbacademy.rag_all`) with a volume and the course PDFs
2. Vector Search endpoint (`vs_all_demo`) with `CAN_USE` for users
3. Read-only user permissions on all shared resources
4. A scheduled **Repair-Resources** job that recreates any shared resource that goes missing

Every step raises on failure, so the deployment stops at the first error and reports it.

**Alerting:** if a step fails, or the final check finds a resource missing, one message is posted to the Slack channel (`#training-automation-alert`) naming the failed step or the missing resources. Nothing is posted when setup succeeds. The Slack webhook URL is read from the Databricks secret `lab-alerts` / `slack-webhook` (see `_lib/slack_alert.py`); if the secret is missing, the alert is skipped with a warning and the run result is unchanged.

## Install Libraries

In [0]:
import sys, os
includes_dir = "."
config_version = "workspace-setup"
sys.path.insert(0, os.path.abspath('.'))
config_path = f"{includes_dir}/config/config-{config_version}.yaml"

from _lib.library_installer import install_libraries
install_libraries(config_path)

In [0]:
dbutils.library.restartPython()

In [0]:
import filecmp
import os
import shutil
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import iam
from databricks.ai_search.client import AISearchClient

ws = WorkspaceClient()
vsc = AISearchClient()

# --- Slack alerting (see _lib/slack_alert.py) ---
import sys
from contextlib import contextmanager
sys.path.insert(0, os.path.abspath("."))
from _lib.slack_alert import send_slack_alert

COURSE_NAME = "Building RAG Agents with Agent Bricks"


@contextmanager
def setup_step(step):
    """Run one setup step. On failure, post a Slack alert naming the step, then re-raise so the run still fails."""
    try:
        yield
    except Exception as e:
        send_slack_alert(
            ws, dbutils,
            title=f"Workspace-Setup FAILED at step: {step}",
            course=COURSE_NAME,
            sections=[("Error", [f"{type(e).__name__}: {e}"])],
        )
        raise

## 1. Define Variables and Create Schema / Volume

Set up the shared UC namespace and define all variable names used throughout this notebook. The `dbacademy` catalog must already exist.

In [0]:
# --- Shared UC namespace ---
SHARED_CATALOG = "dbacademy"
SHARED_SCHEMA  = "rag_all"
SHARED_VOLUME  = "vol_all"

# --- Vector Search ---
VS_ENDPOINT_DEMO = "vs_all_demo"
endpoint_list    = [VS_ENDPOINT_DEMO]

# --- Volume paths ---
VOLUME_PATH          = f"/Volumes/{SHARED_CATALOG}/{SHARED_SCHEMA}/{SHARED_VOLUME}"
LISTINGS_SUBFOLDER   = f"{VOLUME_PATH}/listings"
ADDITIONAL_SUBFOLDER = f"{VOLUME_PATH}/additional_listing_info"
INVOICES_SUBFOLDER   = f"{VOLUME_PATH}/invoices"
LISTINGS_PDF_PATH    = f"{LISTINGS_SUBFOLDER}/airbnb_listings.pdf"
HOUSE_RULES_PDF_PATH = f"{ADDITIONAL_SUBFOLDER}/airbnb_house_rules.pdf"

# --- External (Marketplace) dependencies ---
# Loaded outside this notebook from the Marketplace share; the grant in
# Section 4 is skipped if the table is not present yet.
DATABRICKS_SHARE_NAME = "dbacademy_airbnb"
TABLE_NAME            = "sf_airbnb_listings"

# --- Repair job (keep in sync with Repair-Resources) ---
REPAIR_JOB_NAME      = "Repair Resources"
PREVIOUS_REPAIR_JOB_NAMES = ["Building RAG Agents - Repair Resources"]   # renamed in place on deploy
REPAIR_NOTEBOOK_NAME = "Repair-Resources"
REPAIR_CRON          = "0 */10 * * * ?"   # every 10 minutes, UTC (TESTING ONLY: set back before merging)

# --- Resources from earlier versions of this notebook, removed in Section 5 ---
LEGACY_REFRESH_JOB_NAME = "Workspace-Setup Refresh"
LEGACY_ALERT_NAME       = "Building RAG Agents with Agent Bricks - Workspace-Setup missing resources"

In [0]:
with setup_step("Create schema / volume"):
    spark.sql(f"USE CATALOG {SHARED_CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SHARED_CATALOG}.{SHARED_SCHEMA}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {SHARED_CATALOG}.{SHARED_SCHEMA}.{SHARED_VOLUME}")

print(f"Shared catalog : {SHARED_CATALOG}")
print(f"Shared schema  : {SHARED_SCHEMA}")
print(f"Shared volume  : {VOLUME_PATH}")

## 2. Copy PDFs to Volume

Copy the source PDFs (listings, house rules, invoices) into the shared UC volume so they are accessible to all students. Unchanged files are skipped.

In [0]:
# Source paths (relative to Includes/)
data_dir = os.path.join(os.getcwd(), "data")
invoices_src_dir = os.path.join(data_dir, "invoices")


def copy_file_to_volume(source_file, dest_path):
    """Copy a local file to a UC volume, skipping it if the destination is identical.

    Uses a direct Python copy instead of `dbutils.fs.cp` because on serverless
    compute the latter routes through Spark Connect, which can't read
    `file:/Workspace/...` source paths.
    """
    if not os.path.isfile(source_file):
        raise FileNotFoundError(f"Source PDF not found: {source_file}")
    if os.path.exists(dest_path) and filecmp.cmp(source_file, dest_path, shallow=False):
        print(f"  Skipped (unchanged): {os.path.basename(source_file)} -> {dest_path}")
        return
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    shutil.copyfile(source_file, dest_path)
    print(f"  Copied: {os.path.basename(source_file)} -> {dest_path}")


with setup_step("Copy PDFs to volume"):
    # airbnb_listings.pdf -> vol_all/listings/
    copy_file_to_volume(os.path.join(data_dir, "airbnb_listings.pdf"), LISTINGS_PDF_PATH)

    # airbnb_house_rules.pdf -> vol_all/additional_listing_info/
    copy_file_to_volume(os.path.join(data_dir, "airbnb_house_rules.pdf"), HOUSE_RULES_PDF_PATH)

    # invoices -> vol_all/invoices/
    if os.path.isdir(invoices_src_dir):
        for fname in sorted(os.listdir(invoices_src_dir)):
            if fname.endswith(".pdf"):
                copy_file_to_volume(os.path.join(invoices_src_dir, fname), f"{INVOICES_SUBFOLDER}/{fname}")

print("\nAll PDFs present in volume.")

## 3. Create Vector Search Endpoint

Creates **`vs_all_demo`** (used by demo and lab notebooks) and waits for it to be `ONLINE`. Raises if it is not `ONLINE` after 20 minutes.

In [0]:
def _ensure_vs_endpoint(endpoint_name, timeout_s=20 * 60):
    """Create a VS endpoint if it doesn't exist and wait for ONLINE. Raises on timeout."""
    try:
        vsc.get_endpoint(endpoint_name)
        print(f"Endpoint '{endpoint_name}' already exists.")
    except Exception:
        print(f"Creating endpoint '{endpoint_name}'...")
        vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")

    deadline = time.time() + timeout_s
    while True:
        status = vsc.get_endpoint(endpoint_name).get("endpoint_status", {}).get("state", "")
        if status == "ONLINE":
            print(f"Endpoint '{endpoint_name}' is ONLINE.")
            return
        if time.time() > deadline:
            raise TimeoutError(f"Endpoint '{endpoint_name}' not ONLINE after {timeout_s // 60} min (state={status})")
        print(f"  {endpoint_name} status: {status} - waiting...")
        time.sleep(15)


with setup_step("Create Vector Search endpoint"):
    for ep_name in endpoint_list:
        _ensure_vs_endpoint(ep_name)

### 3a. Grant `CAN_USE` on Endpoints

VS endpoints are workspace objects, so permissions are managed via the Permissions API. Without `CAN_USE`, users cannot query indexes hosted on these endpoints.

In [0]:
with setup_step("Grant CAN_USE on endpoint"):
    for ep_name in endpoint_list:
        ep_id = str(vsc.get_endpoint(ep_name)["id"])
        # update (PATCH) adds this grant without removing other permissions on the endpoint
        ws.permissions.update(
            request_object_type="vector-search-endpoints",
            request_object_id=ep_id,
            access_control_list=[
                iam.AccessControlRequest(group_name="users", permission_level=iam.PermissionLevel.CAN_USE)
            ],
        )
        print(f"Granted CAN_USE on endpoint '{ep_name}' (id={ep_id}) to users.")

## 4. Grant Permissions

Read-only access on the shared namespace and on `system.ai`. `system.ai` can appear a few minutes after a workspace is created, so the notebook waits for it first.

In [0]:
# Wait for system.ai schema to become available before granting permissions
max_attempts = 5
backoff_seconds = 60

with setup_step("Wait for system.ai"):
    for attempt in range(1, max_attempts + 1):
        try:
            spark.sql("DESCRIBE SCHEMA system.ai").collect()  # raises if the schema does not exist
            print(f"system.ai schema is available (attempt {attempt}/{max_attempts}).")
            break
        except Exception as e:
            if attempt == max_attempts:
                raise RuntimeError(f"system.ai not available after {max_attempts} attempts: {e}") from e
            print(f"  Attempt {attempt}/{max_attempts}: system.ai not yet available. Waiting {backoff_seconds}s...")
            time.sleep(backoff_seconds)

In [0]:
# Read-only access on shared catalog resources and system.ai
with setup_step("Grant permissions"):
    for stmt in [
        f"GRANT USE CATALOG ON CATALOG {SHARED_CATALOG} TO `account users`",
        f"GRANT USE SCHEMA  ON SCHEMA  {SHARED_CATALOG}.{SHARED_SCHEMA} TO `account users`",
        f"GRANT READ VOLUME ON VOLUME  {SHARED_CATALOG}.{SHARED_SCHEMA}.{SHARED_VOLUME} TO `account users`",
        "GRANT USE CATALOG ON CATALOG system TO `account users`",
        "GRANT USE SCHEMA ON SCHEMA system.ai TO `account users`",
    ]:
        spark.sql(stmt)
        print(f"OK: {stmt}")

# Marketplace-loaded Airbnb table: grant only if it exists in this workspace
try:
    spark.sql(f"GRANT SELECT ON TABLE {SHARED_CATALOG}.{SHARED_SCHEMA}.{TABLE_NAME} TO `account users`")
    print(f"Granted SELECT on {SHARED_CATALOG}.{SHARED_SCHEMA}.{TABLE_NAME}")
except Exception as e:
    print(
        f"Skipped GRANT on {TABLE_NAME}: {e}\n"
        f"  Load the table from the '{DATABRICKS_SHARE_NAME}' Marketplace share, then re-run this cell."
    )

In [0]:
for stmt in ["SHOW GRANTS ON SCHEMA system.ai", "SHOW GRANTS ON CATALOG system"]:
    display(spark.sql(stmt))

## 5. Deploy Repair-Resources Job

Creates (or updates) a scheduled job that runs `Includes/Repair-Resources`. It checks every shared resource, recreates only what is missing, and fails if a repair does not work.

It also removes the 10-minute `Workspace-Setup Refresh` job and the email alert left by earlier versions of this notebook.

In [0]:
# Never raises: leftover resources from earlier versions must not fail the deployment.
try:
    for job in ws.jobs.list(name=LEGACY_REFRESH_JOB_NAME):
        ws.jobs.delete(job.job_id)
        print(f"Deleted legacy job '{LEGACY_REFRESH_JOB_NAME}' (job_id={job.job_id})")
except Exception as e:
    print(f"WARNING: could not remove legacy job: {e}")

try:
    page_token = None
    while True:
        q = {"page_size": 100, **({"page_token": page_token} if page_token else {})}
        page = ws.api_client.do("GET", "/api/2.0/alerts", query=q)
        for a in page.get("results", []) or page.get("alerts", []):
            if a.get("display_name") == LEGACY_ALERT_NAME and a.get("lifecycle_state", "ACTIVE") == "ACTIVE":
                ws.api_client.do("DELETE", f"/api/2.0/alerts/{a['id']}")
                print(f"Deleted legacy alert '{LEGACY_ALERT_NAME}' (id={a['id']})")
        page_token = page.get("next_page_token")
        if not page_token:
            break
except Exception as e:
    print(f"WARNING: could not remove legacy alert: {e}")

In [0]:
from databricks.sdk.service.jobs import CronSchedule, JobSettings, NotebookTask, PauseStatus, Task

with setup_step("Deploy Repair Resources job"):
    setup_path  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    REPAIR_PATH = f"{os.path.dirname(setup_path)}/{REPAIR_NOTEBOOK_NAME}"

    job_settings = JobSettings(
        name=REPAIR_JOB_NAME,
        tasks=[Task(task_key="repair_resources", notebook_task=NotebookTask(notebook_path=REPAIR_PATH))],
        schedule=CronSchedule(quartz_cron_expression=REPAIR_CRON, timezone_id="UTC", pause_status=PauseStatus.UNPAUSED),
    )

    # Look under earlier names too, so a renamed job is updated in place instead of duplicated
    existing_jobs = [j for n in [REPAIR_JOB_NAME, *PREVIOUS_REPAIR_JOB_NAMES] for j in ws.jobs.list(name=n)]
    if existing_jobs:
        job_id = existing_jobs[0].job_id
        ws.jobs.reset(job_id=job_id, new_settings=job_settings)
        print(f"Updated existing job '{REPAIR_JOB_NAME}' (job_id={job_id})")
    else:
        job_id = ws.jobs.create(name=job_settings.name, tasks=job_settings.tasks, schedule=job_settings.schedule).job_id
        print(f"Created job '{REPAIR_JOB_NAME}' (job_id={job_id})")

    print(f"  Schedule: {REPAIR_CRON} (UTC)")
    print(f"  Notebook: {REPAIR_PATH}")

## 6. Verify Resources

Re-checks every resource. If anything is missing, posts a Slack alert listing it and raises, so the deployment is reported as failed.

In [0]:
missing = []


def _check(name, fn):
    try:
        ok, details = bool(fn()), ""
    except Exception as e:
        ok, details = False, f"{type(e).__name__}: {str(e)[:300]}"
    print(f"  [{'OK' if ok else 'MISSING'}] {name}" + (f" -> {details}" if details else ""))
    if not ok:
        missing.append(f"{name}: {details or 'not found'}")


def _has_grant(sec_type, sec_name, privilege):
    rows = spark.sql(f"SHOW GRANTS ON {sec_type} {sec_name}").collect()
    return any(r["Principal"] == "account users" and r["ActionType"] == privilege for r in rows)


def _endpoint_can_use(name):
    perms = ws.permissions.get("vector-search-endpoints", str(vsc.get_endpoint(name)["id"]))
    allowed = {iam.PermissionLevel.CAN_USE, iam.PermissionLevel.CAN_MANAGE}
    return any(
        acl.group_name == "users" and any(p.permission_level in allowed for p in (acl.all_permissions or []))
        for acl in (perms.access_control_list or [])
    )


vol_full_name = f"{SHARED_CATALOG}.{SHARED_SCHEMA}.{SHARED_VOLUME}"
_check(f"Catalog {SHARED_CATALOG}", lambda: ws.catalogs.get(SHARED_CATALOG))
_check(f"Schema {SHARED_CATALOG}.{SHARED_SCHEMA}", lambda: ws.schemas.get(f"{SHARED_CATALOG}.{SHARED_SCHEMA}"))
_check(f"Volume {vol_full_name}", lambda: ws.volumes.read(vol_full_name))

expected_files = [LISTINGS_PDF_PATH, HOUSE_RULES_PDF_PATH]
if os.path.isdir(invoices_src_dir):
    expected_files += [f"{INVOICES_SUBFOLDER}/{f}" for f in sorted(os.listdir(invoices_src_dir)) if f.endswith(".pdf")]
for path in expected_files:
    _check(path, lambda p=path: os.path.isfile(p))

for ep_name in endpoint_list:
    _check(f"Endpoint {ep_name} ONLINE",
           lambda e=ep_name: vsc.get_endpoint(e).get("endpoint_status", {}).get("state", "") == "ONLINE")
    _check(f"Endpoint {ep_name} CAN_USE -> users", lambda e=ep_name: _endpoint_can_use(e))

for sec_type, sec_name, priv in [
    ("CATALOG", SHARED_CATALOG, "USE CATALOG"),
    ("SCHEMA", f"{SHARED_CATALOG}.{SHARED_SCHEMA}", "USE SCHEMA"),
    ("VOLUME", vol_full_name, "READ VOLUME"),
    ("CATALOG", "system", "USE CATALOG"),
    ("SCHEMA", "system.ai", "USE SCHEMA"),
]:
    _check(f"{priv} ON {sec_type} {sec_name} -> account users",
           lambda t=sec_type, n=sec_name, p=priv: _has_grant(t, n, p))

_check(f"Job {REPAIR_JOB_NAME}", lambda: list(ws.jobs.list(name=REPAIR_JOB_NAME)))

if missing:
    send_slack_alert(
        ws, dbutils,
        title=f"Workspace-Setup FAILED: {len(missing)} resource(s) missing after setup",
        course=COURSE_NAME,
        sections=[("Missing", missing)],
    )
    raise RuntimeError(
        f"Workspace-Setup verification failed: {len(missing)} resource(s) missing:\n"
        + "\n".join(f"  - {m}" for m in missing)
    )
print("All resources present.")

## Summary

In [0]:
print("=" * 60)
print("Workspace Setup Complete")
print("=" * 60)
print(f"  Shared catalog   : {SHARED_CATALOG}")
print(f"  Shared schema    : {SHARED_SCHEMA}")
print(f"  Shared volume    : {VOLUME_PATH}")
print(f"  Listings PDF     : {LISTINGS_PDF_PATH}")
print(f"  VS endpoint demo : {VS_ENDPOINT_DEMO}")
print(f"  Repair job       : {REPAIR_JOB_NAME} ({REPAIR_CRON} UTC)")
print(f"  User permissions : READ-ONLY on shared resources")